In [ ]:
%load_ext autoreload
%autoreload 2

# fast

> Scoring one pattern: its totals from an optional compiled kernel or from the numpy path, and the
> scores reported from them.

In [ ]:
#| default_exp fast

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import os
import numpy as np
import pandas as pd
from randseq.coding import _pattern_is_self_rc, _decode_motif_codes, _motif_occurrences, _motif_totals
from randseq.utils import _motif_pvalues

try:
    if os.environ.get("RANDSEQ_NO_NUMBA"):
        raise ImportError("disabled by RANDSEQ_NO_NUMBA")
    import numba
    from numba import njit, prange
    HAVE_NUMBA = True
except ImportError:
    HAVE_NUMBA = False

## Why a second implementation of the scan

Without numba, one pattern is totalled by the numpy steps of `04_coding` — window codes for every
position of both strands, a sort per row to collapse repeats, and `bincount`s over what is left.
Each step is a full pass over an intermediate array of `n_sequences x 2 x insert length`, and on a
30k-sequence library that is about 0.2 s per pattern. No single step dominates, so no numpy
rewrite of one step buys much.

The kernel here fuses all of it into one compiled pass over the sequences. For each window it
computes the code directly from the 2-bit bases, folds a self-reverse-complementary code onto
`min(code, rc)` with the digit formula, and skips the window if this sequence has already counted
that code — a *stamp* array recording the last sequence seen per code, which replaces the sort.
What it accumulates is exactly what `_motif_totals(*_motif_occurrences(...))` returns, so
`_scan_pattern` can take either path and nothing downstream can tell which ran.

Measured on the 150 nt example library (`counts_panel.csv.gz`, 30,108 inserts after the count
filter) over the 43 shipped patterns: numpy 8.7 s, this
kernel 3.0 s on one core and 0.5 s on eight. The first call in a process also pays a one-off
compile of about 4 s, cached on disk afterwards where the package directory is writable.

### It is optional

numba is not a hard dependency. Install it with the `fast` extra:

```sh
pip install "randseq[fast] @ git+https://github.com/dbikard/randseq.git"
```

Without it, `HAVE_NUMBA` is `False` and the scan runs the numpy path, which is the readable
statement of the algorithm and the reference this kernel is tested against. Setting the
environment variable `RANDSEQ_NO_NUMBA=1` before importing `randseq` forces the numpy path even
when numba is installed.

In [ ]:
#| export
if HAVE_NUMBA:
    @njit(inline="always", cache=True)
    def _rc_code(code, k):
        """Code of the reverse complement: the k base-4 digits read backwards, each 3 - d."""
        r = 0
        for _ in range(k):
            r = (r << 2) | (3 - (code & 3))
            code >>= 2
        return r

    @njit(inline="always", cache=True)
    def _scan_rows(fwd, rc, lo, hi, d1, sp, d2, fc, bg, thr, fold, n, ndep, fsum, bsum, first,
                   stamp):
        """Accumulate the totals of sequences lo..hi-1 into n, ndep, fsum, bsum and first."""
        k = d1 + d2
        w = d1 + sp + d2
        L = fwd.shape[1]
        for i in range(lo, hi):
            dep = 1 if fc[i] < thr else 0
            for strand in range(2):
                arr = fwd if strand == 0 else rc
                for s in range(L - w + 1):
                    code = 0
                    ok = True
                    for j in range(d1):
                        b = arr[i, s + j]
                        if b > 3:               # non-ACGT or padding: the window is rejected
                            ok = False
                            break
                        code = code * 4 + b
                    if ok:
                        for j in range(d1, d1 + sp):   # the spacer is undefined but must
                            if arr[i, s + j] > 3:       # still be ACGT, as in the numpy path
                                ok = False
                                break
                    if ok:
                        for j in range(d2):
                            b = arr[i, s + d1 + sp + j]
                            if b > 3:
                                ok = False
                                break
                            code = code * 4 + b
                    if not ok:
                        continue
                    if fold:
                        r = _rc_code(code, k)
                        if r < code:
                            code = r
                    if stamp[code] != i:        # first time this sequence carries this motif
                        stamp[code] = i
                        if n[code] == 0:
                            first[code] = i
                        n[code] += 1
                        ndep[code] += dep
                        fsum[code] += fc[i]
                        bsum[code] += bg[i]

    @njit(cache=True)
    def _scan_serial(fwd, rc, d1, sp, d2, fc, bg, thr, fold):
        nc = 4 ** (d1 + d2)
        n = np.zeros(nc, np.int64)
        ndep = np.zeros(nc, np.int64)
        fsum = np.zeros(nc, np.float64)
        bsum = np.zeros(nc, np.float64)
        first = np.full(nc, fwd.shape[0], np.int64)
        stamp = np.full(nc, -1, np.int64)
        _scan_rows(fwd, rc, 0, fwd.shape[0], d1, sp, d2, fc, bg, thr, fold, n, ndep, fsum, bsum,
                   first, stamp)
        return n, ndep, fsum, bsum, first

    @njit(parallel=True, cache=True)
    def _scan_parallel(fwd, rc, d1, sp, d2, fc, bg, thr, fold, nchunks):
        # Each chunk of sequences gets its own counters, so threads never write to the same
        # array; the chunks are summed afterwards. A stamp is per chunk too, which is correct
        # because a sequence lives in exactly one chunk.
        nc = 4 ** (d1 + d2)
        nseq = fwd.shape[0]
        N = np.zeros((nchunks, nc), np.int64)
        D = np.zeros((nchunks, nc), np.int64)
        F = np.zeros((nchunks, nc), np.float64)
        B = np.zeros((nchunks, nc), np.float64)
        M = np.full((nchunks, nc), nseq, np.int64)
        for c in prange(nchunks):
            stamp = np.full(nc, -1, np.int64)
            lo = c * nseq // nchunks
            hi = (c + 1) * nseq // nchunks
            _scan_rows(fwd, rc, lo, hi, d1, sp, d2, fc, bg, thr, fold, N[c], D[c], F[c], B[c], M[c],
                       stamp)
        first = np.full(nc, nseq, np.int64)
        for c in range(nchunks):
            for x in range(nc):
                if M[c, x] < first[x]:
                    first[x] = M[c, x]
        return N.sum(axis=0), D.sum(axis=0), F.sum(axis=0), B.sum(axis=0), first

In [ ]:
#| export
def _pattern_totals(encoded_library, pattern, fc_values, background, log2FC_thr=-1, parallel=True):
    """Per motif code: how many sequences carry it, how many of those are depleted, the sum of
    their log2FC, the sum of their background, and the first sequence that carries it.

    The same five arrays as `_motif_totals(*_motif_occurrences(...), ...)`, with the same types,
    computed in one compiled pass. `parallel` spreads the sequences over numba's threads; pass
    False inside a worker process that is itself one of several, so the machine is not asked
    for workers x threads at once.

    Raises if numba is not available -- callers check `HAVE_NUMBA` first.
    """
    if not HAVE_NUMBA:
        raise RuntimeError("numba is not available; install randseq[fast] or use the numpy path")
    d1, sp, d2 = pattern
    if 4 ** (d1 + d2) > np.iinfo(np.int32).max:
        raise ValueError(
            f"pattern {pattern} has {d1 + d2} defined bases; the motif code would overflow "
            f"int32 (max 15). Such a pattern is not usable anyway: it would need "
            f"{4 ** (d1 + d2):.3g} counting bins.")
    fwd, rc = (np.ascontiguousarray(a, dtype=np.uint8) for a in encoded_library)
    fc = np.ascontiguousarray(fc_values, dtype=np.float64)
    bg = np.ascontiguousarray(background, dtype=np.float64)
    fold = _pattern_is_self_rc(pattern)
    threads = numba.get_num_threads()
    # Every chunk holds five counter arrays of 4**(d1+d2) entries, so the chunk count is capped
    # by memory as well as by threads: all chunks together get 2**23 entries per array (~335 MB
    # for the five), reached at nine defined bases by 32 chunks, unless that is fewer than threads.
    nchunks = min(fwd.shape[0], 4 * threads, max(threads, (1 << 23) // 4 ** (d1 + d2)))
    if parallel and threads > 1 and nchunks > 1:
        n, ndep, fsum, bsum, first = _scan_parallel(fwd, rc, d1, sp, d2, fc, bg, float(log2FC_thr),
                                                    fold, nchunks)
    else:
        n, ndep, fsum, bsum, first = _scan_serial(fwd, rc, d1, sp, d2, fc, bg, float(log2FC_thr),
                                                  fold)
    return n, ndep.astype(np.float64), fsum, bsum, first

In [ ]:
from randseq.coding import encode_library, _encode_motif

In [ ]:
if HAVE_NUMBA:
    # the second sequence is the reverse complement of the first
    _enc = encode_library(["GAGACCTT", "AAGGTCTC", "TTTTTTTT"])
    n, n_dep, fc_sum, bg_sum, first = _pattern_totals(_enc, (6, 0, 0), [-3.0, -2.0, 0.5],
                                                     [0.25, 0.5, 0.125], log2FC_thr=-1)
    # (6, 0, 0) is self-rc, so GAGACC and GGTCTC fold onto the smaller code: both of the first two
    # sequences carry it, both are depleted, their backgrounds add, and the first carrier is 0
    _g = _encode_motif("GAGACC", (6, 0, 0))
    print("GAGACC: carriers", n[_g], " depleted", n_dep[_g], " log2FC sum", fc_sum[_g],
          " background sum", bg_sum[_g], " first carrier", first[_g])

GAGACC: carriers 2  depleted 2.0  log2FC sum -5.0  background sum 0.75  first carrier 0


In [ ]:
#| hide
if HAVE_NUMBA:
    test_eq((n[_g], n_dep[_g], fc_sum[_g], bg_sum[_g], first[_g]), (2, 2.0, -5.0, 0.75, 0))
    test_eq(n[_encode_motif("GGTCTC", (6, 0, 0))], 0)       # the larger spelling is never counted
    # TTTTTT folds onto AAAAAA (code 0); it occurs three times per strand in sequence 2 but a
    # motif is counted once per sequence
    test_eq((n[0], n_dep[0], fc_sum[0], bg_sum[0], first[0]), (1, 0.0, 0.5, 0.125, 2))

### What guarantees this is the same scan?

The kernel re-implements every rule the numpy path applies — rejecting a window with a non-ACGT
base or padding, folding the strands of a self-rc pattern, counting a motif once per sequence —
so this notebook's tests check each against `_motif_occurrences` + `_motif_totals`: on the example library
for contiguous, balanced and asymmetric patterns, and on random ragged libraries containing `N`.
`tests/test_fast_scan.py` repeats the comparison over all 43 shipped patterns, and runs the whole
`_get_pattern_scores` both ways. Counts and background sums must agree exactly — `plasmid_background`
returns multiples of 2⁻²⁰, which add exactly in any order — and the log2FC sums to 1e-9, since the
threads add them in a different order.

In [ ]:
#| hide
import random
from randseq.utils import calculate_log2fc, get_lib_seq_context, plasmid_background
from randseq.example_data import get_example_data_dir

In [ ]:
#| hide
if HAVE_NUMBA:

    def _numpy_totals(enc, fc, bg, pat, thr=-1.0):
        si, mc = _motif_occurrences(enc, pat)
        return _motif_totals(si, mc, np.asarray(fc, float), np.asarray(bg, float), thr,
                             4 ** (pat[0] + pat[2]))

    def _same(a, b):
        test_eq(a[0].tolist(), b[0].tolist())                        # support
        test_eq(a[1].tolist(), b[1].tolist())                        # depleted
        assert np.allclose(a[2], b[2], rtol=0, atol=1e-9)            # log2FC sum
        test_eq(a[3].tolist(), b[3].tolist())                        # background sum, exactly
        seen = a[0] > 0
        test_eq(a[4][seen].tolist(), b[4][seen].tolist())            # first sequence

    _c = pd.read_csv(os.path.join(get_example_data_dir(), "countsTable.csv.gz"), index_col=0)
    _l2 = calculate_log2fc(_c[["MFDpir", "JJ1886_T0"]], reference_column="MFDpir")["JJ1886_T0"]
    _enc = encode_library(get_lib_seq_context(_l2.index, "GTCCTAGGTATAATACTAGT",
                                              "GTTTTAGAGCTAGAAATAGC"))
    _bg = plasmid_background(_l2.index, _l2.values, -1)
    # the example library: contiguous, balanced (folded) and asymmetric (not folded) patterns,
    # both the serial kernel and the chunked parallel one
    for _pat in [(6, 0, 0), (5, 0, 0), (4, 4, 4), (3, 4, 3), (4, 4, 3), (2, 1, 3), (4, 8, 4)]:
        for _par in (True, False):
            _same(_pattern_totals(_enc, _pat, _l2.values, _bg, -1, parallel=_par),
                  _numpy_totals(_enc, _l2.values, _bg, _pat))

    _rng = random.Random(0)
    # ragged libraries with N, including N inside the spacer: the rejection rules the example
    # library (pure ACGT, one length) never exercises
    for _ in range(30):
        _seqs = ["".join(_rng.choice("ACGT" if _rng.random() < 0.7 else "ACGTN")
                         for _ in range(_rng.randint(8, 40))) for _ in range(_rng.randint(1, 30))]
        _fc = [_rng.uniform(-6, 2) for _ in _seqs]
        _b = [_rng.randint(1, 2 ** 20 - 1) / 2 ** 20 for _ in _seqs]   # as plasmid_background's
        _pat = _rng.choice([(6, 0, 0), (2, 0, 3), (3, 4, 4), (2, 1, 3), (1, 2, 2), (3, 2, 3), (4, 4, 3)])
        _same(_pattern_totals(encode_library(_seqs), _pat, _fc, _b, -1),
              _numpy_totals(encode_library(_seqs), _fc, _b, _pat))
    print("kernel == numpy path: example library, 7 patterns x serial/parallel, and 30 ragged fuzz libraries")
    # a pattern whose codes would overflow int32 is refused, as in `_pattern_window_codes`
    test_fail(lambda: _pattern_totals(_enc, (16, 0, 0), _l2.values, _bg), contains="overflow")
else:
    print("numba not installed: nothing to compare")

kernel == numpy path: example library, 7 patterns x serial/parallel, and 30 ragged fuzz libraries


## One pattern's scores

`_scan_pattern` returns a pattern's totals from the kernel when numba is installed and from the
numpy steps of `04_coding` otherwise, so every count the pipeline makes of "the plasmids carrying a
motif" comes from one place. `_get_pattern_scores` turns the totals into the reported table:
`fraction_depleted` is depleted carriers over carriers, `avg_log2fc` their mean log2FC, and
`p_value` the binomial test of the depleted count against the carriers' mean background
(`01_utils`).

In [ ]:
#| export
def _scan_pattern(encoded_library, pattern, fc_values, background, log2FC_thr=-1, parallel=True):
    """Per motif code of one pattern: how many sequences carry it, how many of those are
    depleted, the sum of their log2FC, the sum of their background, and the first sequence that
    carries it.

    The compiled kernel (`_pattern_totals`) when numba is installed, `_motif_occurrences` and
    `_motif_totals` otherwise -- the same five arrays either way. Every count the pipeline makes
    of "the plasmids carrying a motif" comes from here, so no two steps can disagree on it.
    `background` is each plasmid's expected depletion rate, from `plasmid_background`.
    """
    fc_values = np.asarray(fc_values, dtype=float)
    background = np.asarray(background, dtype=float)
    if HAVE_NUMBA:
        return _pattern_totals(encoded_library, pattern, fc_values, background, log2FC_thr,
                              parallel=parallel)
    seq_idx, motif_codes = _motif_occurrences(encoded_library, pattern)
    return _motif_totals(seq_idx, motif_codes, fc_values, background, log2FC_thr,
                         4 ** (pattern[0] + pattern[2]))


def _scores_from_totals(totals, pattern, codes):
    """The reported statistics of the given motif codes, from a pattern's totals, ordered by
    first sequence."""
    n_seqs, n_depleted, fc_sum, bg_sum, first_seq = totals
    codes = codes[np.argsort(first_seq[codes], kind="stable")]
    n = n_seqs[codes]
    frac = n_depleted[codes] / n
    return pd.DataFrame({
        'motif': _decode_motif_codes(codes, pattern),
        'fraction_depleted': frac,
        'num_sequences': n,
        'avg_log2fc': fc_sum[codes] / n,
        # P(at least this many of n depleted | null), the null being the carriers' mean
        # background (`plasmid_background`).
        'p_value': _motif_pvalues(n, frac, bg_sum[codes] / n),
    })


def _get_pattern_scores(encoded_library, fc_values, pattern, log2FC_thr=-1,
                       background=None, parallel=True):
    """
    Scores every motif of `pattern` found in the library, on both strands.

    Finds the (sequence, motif) pairs, totals them per motif, and turns the totals into the
    reported statistics. A motif is counted once per sequence that contains it on either strand. `tests/reference_scan.py` holds a readable implementation of the same
    scan, and `tests/test_reference_scan.py` requires the two to give identical tables.

    When numba is installed, the totals come from one compiled pass instead, which returns
    the same arrays; `tests/test_fast_scan.py` requires the two paths to agree.

    Args:
        encoded_library (tuple): (fwd, rc) arrays from `encode_library`.
        fc_values (list or np.ndarray): log2FC of each sequence, in library order.
        pattern (tuple): (d1, spacer, d2) motif pattern.
        log2FC_thr (float): a sequence is depleted when its log2FC is below this value.
        background (np.ndarray): each sequence's expected depletion rate, from
            `plasmid_background`; a motif's p-value is taken against its carriers' mean. Defaults
            to one rate for all, the fraction of this library that is depleted: the encoded
            library holds the sequences with their flanks, not the inserts GC is counted on.
        parallel (bool): let the compiled path use numba's threads. Ignored on the numpy path.

    Returns:
        pd.DataFrame: one row per motif with 'motif', 'fraction_depleted',
            'num_sequences', 'avg_log2fc' and 'p_value', ordered by the first sequence that
            contains the motif.
    """
    fc_values = np.asarray(fc_values, dtype=float)
    if background is None:
        background = np.full(len(fc_values), np.mean(fc_values < log2FC_thr))
    totals = _scan_pattern(encoded_library, pattern, fc_values, background, log2FC_thr,
                          parallel)
    return _scores_from_totals(totals, pattern, np.flatnonzero(totals[0]))

In [ ]:
# The twelve inserts of `04_coding`: GAGACC planted in the four depleted ones
demo_lib = ["GAGACCTTAACG", "TTGAGACCGCAT", "ACGGTCTCAATG", "CCAAGAGACCTT",
            "ACGTTGCAATGC", "TTACGCATTGCA", "GCATTACGGTTA", "ATGCCATTGCAA",
            "CATGCAATTGCG", "TTGCAATGCCAT", "GCAATTGCATGC", "ATTGCCATGCAA"]
demo_fc = np.array([-2.5, -3.0, -3.5, -3.0, 0.2, -0.1, 0.0, 0.3, -0.2, 0.1, 0.0, -0.3])
scores = _get_pattern_scores(encode_library(demo_lib), demo_fc, (6, 0, 0))
scores.sort_values(["fraction_depleted", "num_sequences"], ascending=False).head(6)

,motif,fraction_depleted,num_sequences,avg_log2fc,p_value
6,GAGACC,1.0,4,-3.00,0.012346
0,AAGGTC,1.0,2,-2.75,0.111111
2,AGACCT,1.0,2,-2.75,0.111111
8,AGACCG,1.0,2,-3.25,0.111111
11,GTCTCA,1.0,2,-3.25,0.111111
12,TCTCAA,1.0,2,-3.25,0.111111


The planted site comes out at `fraction_depleted` 1.0 on four plasmids, with a mean log2FC of
−3.0. The motifs below it are 6-mers overlapping the planted site, on two plasmids each: too few
to judge, which is what `min_support` exists to discard.

In [ ]:
#| hide
# The planted site: all four carriers depleted, mean log2FC -3, and one row, not one per strand.
cols = ['fraction_depleted', 'num_sequences', 'avg_log2fc']
_t = scores.set_index('motif')
test_eq(_t.loc['GAGACC', cols].tolist(), [1.0, 4, -3.0])
assert 'GGTCTC' not in _t.index        # folded onto GAGACC
# The same table on the numpy path, whichever path ran above.
_have, HAVE_NUMBA = HAVE_NUMBA, False
_plain = _get_pattern_scores(encode_library(demo_lib), demo_fc, (6, 0, 0))
HAVE_NUMBA = _have
pd.testing.assert_frame_equal(_plain, scores, check_exact=False, rtol=0, atol=1e-12)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()